In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Shadipur, Delhi - CPCB.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,SR,BP,AT,Xylene
0,01-01-2025 00:00,02-01-2025 00:00,139.39,233.33,14.56,60.46,42.67,1.13,15.11,2.53,4.20,2.28,81.05,0.40,256.46,167.71,NaN,NaN,NaN
1,02-01-2025 00:00,03-01-2025 00:00,117.96,184.72,14.33,58.74,41.94,1.01,10.15,2.08,3.24,1.83,87.05,0.68,282.94,184.62,NaN,NaN,NaN
2,03-01-2025 00:00,04-01-2025 00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04-01-2025 00:00,05-01-2025 00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05-01-2025 00:00,06-01-2025 00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,120.41,455.87,26.04,78.11,62.68,1.69,32.53,2.29,4.00,2.84,94.00,0.26,219.49,712.93,NaN,NaN,NaN
316,13-11-2025 00:00,14-11-2025 00:00,128.63,445.23,29.66,86.10,69.87,1.84,25.88,2.29,4.01,2.84,92.66,0.17,209.28,711.26,NaN,NaN,NaN
317,14-11-2025 00:00,15-11-2025 00:00,173.82,402.96,23.38,85.09,64.28,1.58,22.40,2.23,3.83,2.74,93.83,0.19,206.97,657.93,NaN,NaN,NaN
318,15-11-2025 00:00,16-11-2025 00:00,185.13,388.46,30.78,89.21,72.46,1.75,19.67,2.02,3.33,1.98,90.51,0.22,209.37,432.61,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 17)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['BP']
Dropped rows (>70% NaN): 6
Missing values after imputation:
 From Date      0
To Date        0
PM2.5          0
PM10           0
NO             0
NO2            0
NOx            0
CO             0
Ozone          0
Benzene        0
Toluene        0
Eth-Benzene    0
RH             0
WS             0
WD             0
SR             0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (314, 16)
          From Date           To Date   PM2.5    PM10     NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  139.39  233.33  14.56  60.46  42.670   
1  02-01-2025 00:00  03-01-2025 00:00  117.96  184.72  14.33  58.74  41.940   
2  07-01-2025 00:00  08-01-2025 00:00  176.79  246.10  12.66  67.27  45.160   
3  08-01-2025 00:00  09-01-2025 00:00   64.67  285.81  13.81  71.09  48.570   
4  09-01-2025 00:00  10-01-2025 00:00  173.92  390.08  22.19  97.19  44.875   

     CO  Ozone  Benzene  Toluene  Eth-Benzene     RH    WS      WD      SR  
0  1.13  15.11     2.53     4.20         2.28  81.05  0.40  256.46  167.71  
1  1.01  10.15     2.08     3.24         1.83  87.05  0.68  282.94  184.62  
2  1.04  16.93     2.36     4.62         2.08  84.40  0.36  233.08  148.70  
3  1.16  18.87     2.07     3.49         1.62  73.02  0.50  233.30  241.84  
4  1.26  17.75     1.80     2.28         1.32  73.85  0.21  129.18  152.74  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,SR
0,01-01-2025 00:00,02-01-2025 00:00,1.650416,0.790871,-0.003633,-0.316171,-0.231511,1.029474,-0.894892,1.199968,1.900040,1.095846,0.751981,-0.447531,1.405996,-0.893877
1,02-01-2025 00:00,03-01-2025 00:00,1.092952,0.266782,-0.058530,-0.445227,-0.360665,0.150666,-1.484629,0.192808,0.303530,0.063765,1.013369,0.147733,1.873223,-0.822973
2,07-01-2025 00:00,08-01-2025 00:00,2.623312,0.928551,-0.457129,0.194800,0.209029,0.370368,-0.678497,0.819485,2.598513,0.637143,0.897923,-0.532569,0.993466,-0.973586
3,08-01-2025 00:00,09-01-2025 00:00,-0.293295,1.356686,-0.182645,0.481424,0.812339,1.249176,-0.447833,0.170426,0.719288,-0.417872,0.402157,-0.234937,0.997348,-0.583049
4,09-01-2025 00:00,10-01-2025 00:00,2.548654,2.480875,1.817509,2.439769,0.158606,1.981516,-0.581000,-0.433870,-1.292981,-1.105926,0.438316,-0.851460,-0.839802,-0.956646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309,12-11-2025 00:00,13-11-2025 00:00,1.156684,-0.195692,2.736435,1.008151,0.158606,0.077432,1.176322,0.662816,1.567434,2.380212,1.316143,-0.745163,0.753677,1.392232
310,13-11-2025 00:00,14-11-2025 00:00,1.370513,-0.195692,-0.002440,1.607660,0.158606,0.077432,0.385646,0.662816,1.584064,2.380212,1.257767,-0.936498,0.573526,1.385230
311,14-11-2025 00:00,15-11-2025 00:00,2.546053,2.619741,2.101541,1.531877,0.158606,0.077432,-0.028121,0.528528,1.284718,2.150861,1.308737,-0.893979,0.532767,1.161617
312,15-11-2025 00:00,16-11-2025 00:00,2.840263,2.463409,-0.002440,1.841011,0.158606,0.077432,-0.352714,0.058519,0.453203,0.407792,1.164103,-0.830201,0.575114,0.216850


In [10]:
df.to_excel('Shadipur2025.xlsx', index=False)